# SmolLM2-135M — Governed Extraction Fine-Tune (RunPod)

Fine-tunes `HuggingFaceTB/SmolLM2-135M-Instruct` on the 400-example medical-extraction
set distilled from the NIM teacher (`data/medical_extraction_sft.jsonl`), so the tiny
model learns *which* fields to pull and to copy values verbatim — closing the recall gap.

**This notebook trains the extractor only.** At inference it runs behind the AXIOM
governance layer (`axiom_governed_extraction/`), which enforces minimum-necessary
redaction, grounding, egress, and a signed audit trail — so even an imperfect model
cannot leak identifiers or release ungrounded values.

Pick any GPU pod: a 135M full fine-tune runs in a few minutes on a 3090/4090/A40.

## 0. Install dependencies

In [ ]:
%pip install -q -U "trl>=0.12" "transformers>=4.44" datasets accelerate

## 1. Config

In [ ]:
BASE_MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
DATA_PATH  = "medical_extraction_sft.jsonl"   # upload here (drag-drop), or set to a cloned path
OUT_DIR    = "smollm2-135m-medextract"
EPOCHS     = 3
BATCH      = 16
GRAD_ACCUM = 1
LR         = 2e-5
MAX_LEN    = 1024

## 2. Get the training data

**Option A (simplest):** drag-and-drop `medical_extraction_sft.jsonl` into this Jupyter
folder. **Option B:** clone the private repo with a GitHub token (see commented lines).

In [ ]:
import os
# Option B — clone from the private repo with a token:
#   GH_TOKEN = "ghp_xxx"; BRANCH = "claude/governed-extraction"
#   !git clone --depth 1 --branch {BRANCH} https://{GH_TOKEN}@github.com/Orivael-Dev/axiom.git
#   DATA_PATH = "axiom/axiom_governed_extraction/finetune/data/medical_extraction_sft.jsonl"
assert os.path.exists(DATA_PATH), f"Put the dataset at {DATA_PATH} (upload) or set DATA_PATH — see options above."
print("dataset:", DATA_PATH, os.path.getsize(DATA_PATH), "bytes")

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files=DATA_PATH, split="train")
print(ds)
print("USER  :", ds[0]["messages"][1]["content"][:160].replace("\n", " "))
print("TARGET:", ds[0]["messages"][2]["content"][:160])

## 3. Load base model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16 if BF16 else torch.float16)
print(f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M params | bf16={BF16}")

## 4. Train (SFT)

TRL applies the tokenizer chat template to the `messages` field automatically. The
try/except wrappers keep this working across TRL versions (arg renames).

In [ ]:
from trl import SFTTrainer, SFTConfig

common = dict(
    output_dir=OUT_DIR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, warmup_ratio=0.03, lr_scheduler_type="cosine",
    logging_steps=10, save_strategy="epoch", bf16=BF16, fp16=not BF16,
    report_to="none",
)
try:
    cfg = SFTConfig(max_length=MAX_LEN, packing=False, **common)
except TypeError:
    cfg = SFTConfig(max_seq_length=MAX_LEN, packing=False, **common)

try:
    trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds, processing_class=tok)
except TypeError:
    trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds, tokenizer=tok)

trainer.train()

In [ ]:
trainer.save_model(OUT_DIR)
tok.save_pretrained(OUT_DIR)
print("saved ->", OUT_DIR)

## 5. Quick eval — does it extract the right fields now?

Raw extraction quality (the governance layer wraps this downstream). Compare against
the un-fine-tuned baseline, which grabbed identifiers and missed clinical fields.

In [ ]:
FIELDS = ["visit_date","diagnosis","medications","procedure","lab_values",
          "patient_name","mrn","ssn","dob","address","phone"]
SYS = ("Extract medical fields from the document. Output ONLY a JSON object. "
       "Keys must be from this list and nothing else: " + ", ".join(FIELDS) + ". "
       "Each value is the exact text copied from the document, or omit the key if absent. "
       "Do not add commentary, do not repeat keys.")
doc = ("DISCHARGE SUMMARY\nPatient: Maria Gonzalez\nMRN: A4471-9920\nDOB: 1971-03-14\n"
       "Visit date: 2026-02-09\nDiagnosis: Type 2 diabetes mellitus with poor glycemic control\n"
       "Medications: Metformin 1000 mg BID, Lisinopril 10 mg daily\n"
       "Procedure: point-of-care glucose testing")
msgs = [{"role": "system", "content": SYS},
        {"role": "user", "content": "DOCUMENT:\n" + doc + "\n\nJSON:"}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
out = model.generate(ids, max_new_tokens=256, do_sample=False, pad_token_id=tok.pad_token_id)
print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

## 6. (Optional) Export to GGUF

For the on-device / `llama.cpp` backend. `q8_0` is produced directly here; `Q4_K_M`
or SRD quantization is a local `llama-quantize` step on the f16 GGUF.

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
%pip install -q -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py {OUT_DIR} --outfile medextract-f16.gguf  --outtype f16
!python llama.cpp/convert_hf_to_gguf.py {OUT_DIR} --outfile medextract-q8_0.gguf --outtype q8_0
print("GGUF ready. For Q4_K_M or SRD, run llama-quantize on medextract-f16.gguf locally.")

## 7. (Optional) Keep the artifacts

Download via the Jupyter file browser, or push to the Hub.

In [ ]:
# Download: right-click medextract-q8_0.gguf (or zip OUT_DIR) in the file browser -> Download.
# Or push to the Hub:
# from huggingface_hub import login, upload_folder
# login("hf_xxx")
# upload_folder(folder_path=OUT_DIR, repo_id="YOURNAME/smollm2-135m-medextract")

### Next: run it under governance

Drop `medextract-q8_0.gguf` (or your Q4_K_M) into `llama-server` and point the
governed pipeline's `LlamaCppBackend` at it:

```bash
llama-server -m medextract-q8_0.gguf -c 2048 --port 8080 --no-webui
python run_demo.py --local
```

Re-measure: authorized-field recall should rise sharply; identifier-leak rate stays 0
(governance), and fabrication-blocks fall as the model stops mangling values.